# Notebook de Simulação e Teste para o Engine

Este notebook permite executar o fluxo de trabalho de dois componentes principais:
1. **Engine (Passo a Passo)**: Para um único ativo escolhido, ideal para depuração detalhada.

**Pré-requisitos:**
1. O terminal MetaTrader 5 deve estar aberto e logado.
2. Os modelos de produção devem ter sido gerados pelo script `train_model.py`.

## 1. Importações e Configuração Inicial

Importamos as bibliotecas necessárias e configuramos o ambiente de trabalho.

In [1]:
import sys
from pathlib import Path
from datetime import datetime
import pandas as pd

# Adiciona o diretório raiz ao path para importar os módulos do projeto
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.simulation.engine import SimulationEngine

print(f"[OK] Importacoes realizadas com sucesso!")
print(f"[OK] Diretorio do projeto: {project_root}")

2025-11-02 08:15:50,476 - INFO - Diretório de cache de dados inicializado em: C:\projects\wtnps-trade\.cache_data


[OK] Importacoes realizadas com sucesso!
[OK] Diretorio do projeto: c:\projects\wtnps-trade


## 2. Inicialização do SimulationEngine

Criamos uma instância do `SimulationEngine` que carrega automaticamente as configurações do arquivo `configs/main.yaml`.

In [2]:
# Inicializa o engine apontando para o arquivo de configuração
config_path = project_root / 'configs' / 'main.yaml'
engine = SimulationEngine(config_path=str(config_path))

print(f"[OK] SimulationEngine inicializado!")
print(f"[OK] Timezone local: {engine.local_tz_str}")
print(f"[OK] Diretorio de modelos: {engine.models_dir}")

2025-11-02 08:15:58,177 - INFO - Carregando config: c:\projects\wtnps-trade\configs\main.yaml
2025-11-02 08:15:58,187 - INFO - Models dir resolvido para: C:\projects\wtnps-trade\models
2025-11-02 08:15:58,188 - INFO - Timezone padrão definido para UTC.


[OK] SimulationEngine inicializado!
[OK] Timezone local: UTC
[OK] Diretorio de modelos: C:\projects\wtnps-trade\models


## 3. Definir Parâmetros da Simulação

Configure os parâmetros para a simulação de um ciclo único:
- **asset_symbol**: O ticker do ativo (ex: 'WDO$', 'WIN$')
- **timeframe_str**: O timeframe desejado (ex: 'H1', 'M15', 'D1')
- **target_datetime_local**: Data e hora local (São Paulo) para simular

In [3]:
# Parâmetros da simulação
asset_symbol = 'WDO$'  # Altere para o ativo desejado
timeframe_str = 'M5'   # Timeframes válidos: M1, M5, M15, M30, H1, H4, D1, W1, MN1
target_datetime_local = datetime(2025, 10, 31, 9, 0, 0)  # Data/hora local (São Paulo)

print(f"[OK] Parametros configurados:")
print(f"  - Ativo: {asset_symbol}")
print(f"  - Timeframe: {timeframe_str}")
print(f"  - Data/Hora (local): {target_datetime_local}")

[OK] Parametros configurados:
  - Ativo: WDO$
  - Timeframe: M5
  - Data/Hora (local): 2025-10-31 09:00:00


## 4. Executar Ciclo de Simulação

Executa um único ciclo de simulação usando `run_simulation_cycle()`. Este método:
1. Carrega os recursos do ativo (modelo, estratégia, scaler)
2. Converte o horário local para UTC
3. Busca os dados de mercado via provedor configurado
4. Prepara as features e gera o sinal da IA
5. Valida o sinal contra as regras de setup
6. Retorna um dicionário com todos os resultados

In [4]:
# Executa o ciclo de simulação
result = engine.run_simulation_cycle(
    asset_symbol=asset_symbol,
    timeframe_str=timeframe_str,
    target_datetime_local=target_datetime_local
)

print(f"[OK] Ciclo de simulacao executado!")
print(f"[OK] Status: {result.get('status', 'N/A')}")

2025-11-02 08:16:14,162 - INFO - Iniciando ciclo simulação: WDO$ @ M5 em 2025-10-31 09:00 UTC


KeyboardInterrupt: 

## 5. Análise de Dados OHLC Históricos

Busca e exibe os dados OHLC (Open, High, Low, Close) anteriores ao target date, incluindo os indicadores técnicos calculados (EMA9, SMA20, SMA50, SMA200).

In [ ]:
from datetime import timedelta
import pytz

# Busca dados históricos (últimos 50 candles antes do target)
asset_config = None
for cfg in engine.config.get('assets', []):
    if cfg.get('ticker') == asset_symbol:
        asset_config = cfg
        break

if asset_config:
    provider_name = asset_config.get('provider', 'MetaTrader5')
    
    # Converte target_datetime_local para UTC
    local_tz = pytz.timezone('America/Sao_Paulo')
    target_dt_local_aware = local_tz.localize(target_datetime_local)
    target_dt_utc = target_dt_local_aware.astimezone(pytz.utc)
    
    # Busca 50 candles antes do target (ajuste conforme timeframe)
    if timeframe_str == 'H1':
        lookback_hours = 50
    elif timeframe_str == 'M15':
        lookback_hours = 50 * 0.25
    elif timeframe_str == 'D1':
        lookback_hours = 50 * 24
    else:
        lookback_hours = 50  # default
    
    start_dt_utc = target_dt_utc - timedelta(hours=lookback_hours)
    
    # Busca dados via engine
    historical_data = engine._get_market_data(
        ticker=asset_symbol,
        start_dt_utc=start_dt_utc,
        end_dt_utc=target_dt_utc,
        timeframe_str=timeframe_str,
        provider_name=provider_name
    )
    
    if not historical_data.empty:
        # Calcula indicadores técnicos
        historical_data['EMA9'] = historical_data['close'].ewm(span=9, adjust=False).mean()
        historical_data['SMA20'] = historical_data['close'].rolling(window=20).mean()
        historical_data['SMA50'] = historical_data['close'].rolling(window=50).mean()
        historical_data['SMA200'] = historical_data['close'].rolling(window=200).mean()
        
        print(f"[OK] Dados historicos carregados: {len(historical_data)} candles")
        print(f"  Periodo: {historical_data.index[0]} ate {historical_data.index[-1]}")
        print("\n[DADOS] Ultimos 10 candles com indicadores:")
        print("-" * 120)
        
        # Exibe últimos 10 candles com indicadores
        display_cols = ['open', 'high', 'low', 'close', 'EMA9', 'SMA20', 'SMA50', 'SMA200']
        print(historical_data[display_cols].tail(10).to_string())
        print("-" * 120)
    else:
        print("[AVISO] Nenhum dado historico encontrado para o periodo especificado.")
        historical_data = pd.DataFrame()
else:
    print(f"[AVISO] Configuracao nao encontrada para {asset_symbol}.")
    historical_data = pd.DataFrame()

## 6. Visualização Gráfica - Preço e Indicadores

Gráfico interativo mostrando a evolução do preço de fechamento com os indicadores técnicos (EMA9, SMA20, SMA50, SMA200).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

if not historical_data.empty:
    fig, ax = plt.subplots(figsize=(15, 8))
    
    # Plot preço de fechamento
    ax.plot(historical_data.index, historical_data['close'], 
            label='Preco Fechamento', color='black', linewidth=2, alpha=0.8)
    
    # Plot indicadores
    ax.plot(historical_data.index, historical_data['EMA9'], 
            label='EMA 9', color='blue', linewidth=1.5, alpha=0.7)
    ax.plot(historical_data.index, historical_data['SMA20'], 
            label='SMA 20', color='orange', linewidth=1.5, alpha=0.7)
    ax.plot(historical_data.index, historical_data['SMA50'], 
            label='SMA 50', color='green', linewidth=1.5, alpha=0.7)
    ax.plot(historical_data.index, historical_data['SMA200'], 
            label='SMA 200', color='red', linewidth=1.5, alpha=0.7)
    
    # Marca o ponto do target datetime
    if not historical_data.empty:
        target_close = historical_data.iloc[-1]['close']
        ax.scatter(historical_data.index[-1], target_close, 
                  color='red', s=100, zorder=5, label='Target Date')
        ax.axvline(x=historical_data.index[-1], color='red', 
                  linestyle='--', alpha=0.5)
    
    # Configurações do gráfico
    ax.set_xlabel('Data/Hora', fontsize=12, fontweight='bold')
    ax.set_ylabel('Preco', fontsize=12, fontweight='bold')
    ax.set_title(f'{asset_symbol} - Preco e Indicadores Tecnicos ({timeframe_str})', 
                fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Formata eixo X para datas
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    print(f"[OK] Grafico gerado com sucesso!")
else:
    print("[AVISO] Sem dados para plotar.")

## 7. Gráfico de Candlestick com Volume

Visualização em candlestick dos últimos candles com volume de negociação.

In [ ]:
import mplfinance as mpf

if not historical_data.empty:
    # Prepara dados para mplfinance (últimos 30 candles)
    plot_data = historical_data.tail(30).copy()
    
    # Renomeia colunas para o formato esperado pelo mplfinance
    plot_data_mpf = plot_data.rename(columns={
        'open': 'Open',
        'high': 'High', 
        'low': 'Low',
        'close': 'Close',
        'volume': 'Volume'
    })
    
    # Adiciona médias móveis ao gráfico
    apds = [
        mpf.make_addplot(plot_data['EMA9'], color='blue', width=1.5, label='EMA9'),
        mpf.make_addplot(plot_data['SMA20'], color='orange', width=1.5, label='SMA20'),
        mpf.make_addplot(plot_data['SMA50'], color='green', width=1.5, label='SMA50'),
    ]
    
    # Cria o gráfico candlestick
    mpf.plot(
        plot_data_mpf,
        type='candle',
        style='charles',
        title=f'{asset_symbol} - Candlestick ({timeframe_str}) - Ultimos 30 Candles',
        ylabel='Preco',
        volume=True,
        ylabel_lower='Volume',
        figsize=(15, 10),
        addplot=apds,
        warn_too_much_data=len(plot_data_mpf) + 1
    )
    
    print(f"[OK] Grafico candlestick gerado com sucesso!")
else:
    print("[AVISO] Sem dados para plotar.")

## 8. Resultado da Simulação - Visão Geral

Exibe as informações principais do resultado da simulação.

## 5. Analisar Resultados - Visão Geral

Exibe as informações principais do resultado da simulação.

In [ ]:
print("=" * 70)
print(f"RESULTADO DA SIMULAÇÃO - {asset_symbol} @ {timeframe_str}")
print("=" * 70)
print(f"Timestamp Local: {result.get('timestamp_local')}")
print(f"Timestamp UTC:   {result.get('timestamp_utc')}")
print(f"Status:          {result.get('status')}")
print()
print(f"Sinal da IA:     {result.get('ai_signal')}")
print(f"Setup Válido:    {result.get('setup_valid')}")
print(f"Decisão Final:   {result.get('final_decision')}")
print()
print(f"Preço de Fechamento: {result.get('close_price')}")
print(f"Preço Sugerido:      {result.get('suggested_price')}")
print(f"Stop Loss:           {result.get('stop_loss')}")
print(f"Take Profit:         {result.get('take_profit')}")
print("=" * 70)

## 6. Analisar Indicadores Técnicos

Exibe os valores dos indicadores técnicos calculados pela estratégia.

In [ ]:
indicators = result.get('indicators', {})

if indicators:
    print("\nINDICADORES TECNICOS:")
    print("-" * 50)
    for key, value in indicators.items():
        # Formata valores numéricos com 4 casas decimais
        if isinstance(value, (int, float)):
            print(f"  {key:20s}: {value:12.4f}")
        else:
            print(f"  {key:20s}: {value}")
    print("-" * 50)
else:
    print("[AVISO] Nenhum indicador disponivel no resultado.")

## 7. Detalhes da Validação de Setup

Exibe os detalhes da validação das regras de setup técnico.

In [ ]:
setup_details = result.get('setup_details', {})

if setup_details:
    print("\nDETALHES DA VALIDACAO DE SETUP:")
    print("-" * 50)
    
    rules_checked = setup_details.get('rules_checked', [])
    if rules_checked:
        print(f"Total de regras verificadas: {len(rules_checked)}")
        for i, rule in enumerate(rules_checked, 1):
            status = "[OK] PASSOU" if rule.get('passed') else "[X] FALHOU"
            print(f"\n  Regra {i}: {rule.get('type')} ({rule.get('condition')})")
            print(f"    Status: {status}")
            print(f"    Motivo: {rule.get('reason', 'N/A')}")
    else:
        print("  Nenhuma regra de setup configurada (valido por padrao).")
    
    print("-" * 50)
else:
    print("[AVISO] Nenhum detalhe de setup disponivel no resultado.")

## 8. Resultado Completo (JSON)

Visualiza o resultado completo em formato estruturado para análise detalhada.

In [ ]:
import json

# Converte datetime para string para exibição JSON
result_json = result.copy()
if 'timestamp_local' in result_json and hasattr(result_json['timestamp_local'], 'isoformat'):
    result_json['timestamp_local'] = result_json['timestamp_local'].isoformat()
if 'timestamp_utc' in result_json and hasattr(result_json['timestamp_utc'], 'isoformat'):
    result_json['timestamp_utc'] = result_json['timestamp_utc'].isoformat()

print("\nRESULTADO COMPLETO:")
print(json.dumps(result_json, indent=2, ensure_ascii=False, default=str))

## 9. Simulação de Resultado de Trade

Simula o resultado financeiro executando a sugestão recebida na simulação. Configure os parâmetros de entrada e saída do trade para calcular lucro/prejuízo e métricas relevantes.

In [ ]:
# Parâmetros do trade baseados na simulação
trade_direction = result.get('final_decision', 'HOLD')  # COMPRA ou VENDA
entry_price = result.get('suggested_price', result.get('close_price', 0))
stop_loss = result.get('stop_loss', 0)
take_profit = result.get('take_profit', 0)

# Parâmetros de saída do trade (você pode ajustar manualmente)
# Opção 1: Saída no Stop Loss ou Take Profit (simular o pior/melhor cenário)
# Opção 2: Saída em um preço específico
exit_scenario = 'take_profit'  # Opções: 'take_profit', 'stop_loss', 'custom'
exit_price_custom = None  # Use se exit_scenario = 'custom'

# Configuração do trade
contracts = 1  # Número de contratos
contract_size = 1  # Tamanho do contrato (pontos por contrato)

# Calcula preço de saída
if exit_scenario == 'take_profit':
    exit_price = take_profit
    scenario_name = "Take Profit"
elif exit_scenario == 'stop_loss':
    exit_price = stop_loss
    scenario_name = "Stop Loss"
elif exit_scenario == 'custom' and exit_price_custom:
    exit_price = exit_price_custom
    scenario_name = "Preco Customizado"
else:
    exit_price = entry_price
    scenario_name = "Sem Saida"

# Calcula resultado do trade
if trade_direction == 'COMPRA':
    price_diff = exit_price - entry_price
elif trade_direction == 'VENDA':
    price_diff = entry_price - exit_price
else:
    price_diff = 0

profit_loss = price_diff * contracts * contract_size

# Calcula métricas adicionais
if entry_price > 0:
    roi_percent = (profit_loss / entry_price) * 100
    risk_reward = abs((take_profit - entry_price) / (entry_price - stop_loss)) if stop_loss != entry_price else 0
else:
    roi_percent = 0
    risk_reward = 0

print("=" * 80)
print(f"SIMULACAO DE RESULTADO DO TRADE - {asset_symbol}")
print("=" * 80)
print(f"\n[DADOS] PARAMETROS DO TRADE:")
print(f"  Direcao:            {trade_direction}")
print(f"  Preco de Entrada:   {entry_price:.2f}")
print(f"  Stop Loss:          {stop_loss:.2f}")
print(f"  Take Profit:        {take_profit:.2f}")
print(f"  Contratos:          {contracts}")
print(f"\n[CENARIO] SAIDA: {scenario_name}")
print(f"  Preco de Saida:     {exit_price:.2f}")
print(f"  Diferenca de Pontos: {price_diff:.2f}")
print(f"\n[RESULTADO] FINANCEIRO:")
print(f"  Lucro/Prejuizo:     {profit_loss:+.2f} pontos")
print(f"  ROI:                {roi_percent:+.2f}%")
print(f"  Risk/Reward Ratio:  1:{risk_reward:.2f}")
print("=" * 80)

## 10. Gráfico de Análise do Trade

Visualização gráfica do trade simulado com entrada, stop loss, take profit e saída.

In [ ]:
if not historical_data.empty:
    fig, ax = plt.subplots(figsize=(15, 8))
    
    # Plot preço de fechamento
    ax.plot(historical_data.index, historical_data['close'], 
            label='Preco Fechamento', color='black', linewidth=2, alpha=0.8)
    
    # Plot EMA9 e SMA20 (indicadores mais relevantes)
    ax.plot(historical_data.index, historical_data['EMA9'], 
            label='EMA 9', color='blue', linewidth=1.5, alpha=0.6, linestyle='--')
    ax.plot(historical_data.index, historical_data['SMA20'], 
            label='SMA 20', color='orange', linewidth=1.5, alpha=0.6, linestyle='--')
    
    # Marca o ponto de entrada
    entry_idx = historical_data.index[-1]
    ax.scatter(entry_idx, entry_price, color='green', s=200, 
              zorder=5, marker='^', label=f'Entrada {trade_direction}', edgecolors='darkgreen', linewidths=2)
    
    # Linha de entrada
    ax.axhline(y=entry_price, color='green', linestyle='--', alpha=0.5, linewidth=1.5)
    
    # Linhas de Stop Loss e Take Profit
    ax.axhline(y=stop_loss, color='red', linestyle='--', alpha=0.7, linewidth=2, label=f'Stop Loss ({stop_loss:.2f})')
    ax.axhline(y=take_profit, color='blue', linestyle='--', alpha=0.7, linewidth=2, label=f'Take Profit ({take_profit:.2f})')
    
    # Marca ponto de saída
    if exit_price != entry_price:
        exit_color = 'green' if profit_loss > 0 else 'red'
        exit_marker = 'v' if profit_loss > 0 else 'x'
        ax.scatter(entry_idx, exit_price, color=exit_color, s=200, 
                  zorder=5, marker=exit_marker, label=f'Saida ({scenario_name})', 
                  edgecolors='darkred' if profit_loss < 0 else 'darkgreen', linewidths=2)
        
        # Desenha linha do trade
        ax.annotate('', xy=(entry_idx, exit_price), xytext=(entry_idx, entry_price),
                   arrowprops=dict(arrowstyle='<->', color=exit_color, lw=2, alpha=0.7))
        
        # Adiciona texto com resultado
        mid_price = (entry_price + exit_price) / 2
        result_text = f'{profit_loss:+.2f} pts\n{roi_percent:+.1f}%'
        ax.text(entry_idx, mid_price, result_text, fontsize=10, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.5', facecolor=exit_color, alpha=0.3),
               ha='left', va='center')
    
    # Configurações do gráfico
    ax.set_xlabel('Data/Hora', fontsize=12, fontweight='bold')
    ax.set_ylabel('Preco', fontsize=12, fontweight='bold')
    title_color = 'green' if profit_loss > 0 else 'red' if profit_loss < 0 else 'gray'
    ax.set_title(f'Analise do Trade - {asset_symbol} ({timeframe_str}) | Resultado: {profit_loss:+.2f} pts ({roi_percent:+.1f}%)', 
                fontsize=14, fontweight='bold', color=title_color)
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Formata eixo X
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    print(f"[OK] Grafico de analise do trade gerado!")
else:
    print("[AVISO] Sem dados para plotar.")

## 11. Indicadores Mais Relevantes do Trade

Apresenta um resumo consolidado dos indicadores técnicos mais importantes no momento da entrada do trade.

In [ ]:
if not historical_data.empty:
    # Pega os valores dos indicadores no momento da entrada
    last_candle = historical_data.iloc[-1]
    
    # Calcula indicadores adicionais
    price_vs_ema9 = ((last_candle['close'] - last_candle['EMA9']) / last_candle['EMA9']) * 100
    price_vs_sma20 = ((last_candle['close'] - last_candle['SMA20']) / last_candle['SMA20']) * 100
    price_vs_sma50 = ((last_candle['close'] - last_candle['SMA50']) / last_candle['SMA50']) * 100
    price_vs_sma200 = ((last_candle['close'] - last_candle['SMA200']) / last_candle['SMA200']) * 100
    
    # Identifica tendência pelas médias
    if last_candle['EMA9'] > last_candle['SMA20'] > last_candle['SMA50']:
        trend = "[ALTA] Forte"
    elif last_candle['EMA9'] > last_candle['SMA20']:
        trend = "[ALTA] Moderada"
    elif last_candle['EMA9'] < last_candle['SMA20'] < last_candle['SMA50']:
        trend = "[BAIXA] Forte"
    elif last_candle['EMA9'] < last_candle['SMA20']:
        trend = "[BAIXA] Moderada"
    else:
        trend = "[LATERAL]"
    
    # Volatilidade (amplitude do candle)
    candle_range = last_candle['high'] - last_candle['low']
    volatility_pct = (candle_range / last_candle['close']) * 100
    
    print("=" * 80)
    print(f"INDICADORES MAIS RELEVANTES - {asset_symbol} @ {target_datetime_local}")
    print("=" * 80)
    print(f"\n[DADOS] PRECOS E MEDIAS MOVEIS:")
    print(f"  Preco Fechamento:   {last_candle['close']:.2f}")
    print(f"  EMA 9:              {last_candle['EMA9']:.2f}  ({price_vs_ema9:+.2f}%)")
    print(f"  SMA 20:             {last_candle['SMA20']:.2f}  ({price_vs_sma20:+.2f}%)")
    print(f"  SMA 50:             {last_candle['SMA50']:.2f}  ({price_vs_sma50:+.2f}%)")
    print(f"  SMA 200:            {last_candle['SMA200']:.2f}  ({price_vs_sma200:+.2f}%)")
    
    print(f"\n[TENDENCIA] ANALISE:")
    print(f"  Tendencia Detectada: {trend}")
    print(f"  Alinhamento EMA9/SMA20: {'[OK] Alinhado' if (last_candle['EMA9'] > last_candle['SMA20'] and trade_direction == 'COMPRA') or (last_candle['EMA9'] < last_candle['SMA20'] and trade_direction == 'VENDA') else '[X] Contrario'}")
    
    print(f"\n[VOLATILIDADE]:")
    print(f"  Range do Candle:    {candle_range:.2f} pts")
    print(f"  Volatilidade:       {volatility_pct:.2f}%")
    
    print(f"\n[TRADE] PARAMETROS:")
    print(f"  Risk/Reward Ratio:  1:{risk_reward:.2f}")
    print(f"  Distancia ao Stop:  {abs(entry_price - stop_loss):.2f} pts ({abs((entry_price - stop_loss) / entry_price * 100):.2f}%)")
    print(f"  Distancia ao Take:  {abs(take_profit - entry_price):.2f} pts ({abs((take_profit - entry_price) / entry_price * 100):.2f}%)")
    
    # Adiciona indicadores da estratégia
    strategy_indicators = result.get('indicators', {})
    if strategy_indicators:
        print(f"\n[ESTRATEGIA] INDICADORES:")
        relevant_indicators = ['RSI', 'MACD', 'signal_line', 'ATR', 'BB_upper', 'BB_lower', 'volume']
        for ind_name in relevant_indicators:
            if ind_name in strategy_indicators:
                value = strategy_indicators[ind_name]
                if isinstance(value, (int, float)):
                    print(f"  {ind_name:15s}: {value:.4f}")
    
    print("=" * 80)
else:
    print("[AVISO] Sem dados historicos para analise de indicadores.")

---

## 📝 Notas Importantes

### Dependências Adicionais
Este notebook utiliza bibliotecas de visualização que podem não estar instaladas:
```powershell
poetry add matplotlib mplfinance
```

### Interpretação dos Resultados
- **Risk/Reward > 2.0**: Trade com boa relação risco/retorno
- **Alinhamento de Tendência**: Confirma se o trade está a favor da tendência
- **Volatilidade Alta (> 2%)**: Pode indicar maior risco e oportunidade
- **Indicadores Estratégicos**: RSI, MACD e ATR fornecem confirmação adicional

## 9. Finalizar - Fechar Engine

Libera os recursos utilizados pelo engine (fechar conexões com provedores de dados).

In [ ]:
# Fecha todas as conexões e libera recursos
engine.close()

print("[OK] Engine finalizado com sucesso!")
print("[OK] Recursos liberados.")

---

## Dicas de Uso

### Múltiplas Simulações
Para executar múltiplas simulações, basta alterar os parâmetros na célula 3 e executar novamente as células 4-8.

### Análise de Diferentes Ativos
Altere o `asset_symbol` para testar diferentes ativos configurados em `configs/main.yaml` (ex: 'WIN$', 'PETR4').

### Diferentes Timeframes
Experimente diferentes timeframes: M1, M5, M15, M30, H1, H4, D1, W1, MN1.

### Troubleshooting
- **Erro de modelo não encontrado**: Execute `poetry run python train_model.py` primeiro
- **Erro de conexão MT5**: Certifique-se de que o MetaTrader 5 está aberto e logado
- **Dados vazios**: Verifique se a data/hora está dentro do período de dados disponível